<a href="https://colab.research.google.com/github/Aman-Semwal/Celebal-Data-Science-Assignments/blob/main/week8_aman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### Implementation
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling


In [8]:
# Imports (used for routing, safe eval, and logging)

import re
import ast
import operator
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("agent")

In [9]:
# TOOL 1: Calculator

# Safe operators allowed in the calculator (avoids using raw eval on user input)
_ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
    ast.Mod: operator.mod,
}


def _safe_eval(node):
    """Recursively evaluate a parsed AST expression using only whitelisted operators."""
    if isinstance(node, ast.Constant):  # numbers
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numeric constants are allowed")
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Unsupported expression")


def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely (no arbitrary eval)."""
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_eval(tree.body)
        return str(result)
    except Exception:
        return "Error in calculation"

In [10]:
# TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower().strip(".,!?") for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

In [11]:
# TOOL 3 (Bonus): Word / Character Counter

def word_counter(text: str) -> dict:
    """Count words and characters in a piece of text."""
    try:
        words = text.split()
        return {"word_count": len(words), "char_count": len(text)}
    except Exception:
        return {"word_count": 0, "char_count": 0}

## Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- If query contains "count" → use word counter (bonus tool)
- Else → general response

In [12]:
# AGENT FUNCTION

def _extract_math_expression(query: str) -> str:
    """Pull just the numeric/operator part out of a natural-language calculate query."""
    match = re.search(r"[-+]?[\d\s\.\+\-\*\/\(\)]+", query)
    return match.group().strip() if match else ""


def agent(query: str):
    """Single-agent router: understands intent, calls the right tool, returns structured JSON."""

    if not isinstance(query, str) or not query.strip():
        logger.warning("Received empty or invalid query.")
        return {"type": "error", "result": "Query must be a non-empty string."}

    query_lower = query.lower()
    logger.info(f"Received query: {query!r}")

    try:
        # --- Route 1: Math / calculation queries ---
        if "calculate" in query_lower or re.search(r"\d+\s*[\+\-\*/]\s*\d+", query):
            expression = _extract_math_expression(query)
            if not expression:
                logger.error("No valid expression found for calculation.")
                return {"type": "error", "result": "No valid mathematical expression found."}

            logger.info(f"Routing to Calculator Tool with expression: {expression!r}")
            result = calculator(expression)

            if result == "Error in calculation":
                return {"type": "error", "result": result}
            return {"type": "calculation", "result": result}

        # --- Route 2: Keyword extraction queries ---
        elif "keyword" in query_lower:
            logger.info("Routing to Keyword Extraction Tool.")
            keywords = extract_keywords(query)
            return {"type": "keywords", "result": keywords}

        # --- Route 3 (Bonus): Word / character count queries ---
        elif "count" in query_lower:
            logger.info("Routing to Word Counter Tool.")
            counts = word_counter(query)
            return {"type": "count", "result": counts}

        # --- Route 4: General / fallback queries ---
        else:
            logger.info("Routing to General Response.")
            return {
                "type": "general",
                "result": f"I don't have a specific tool for this, but here's a direct response to: '{query}'"
            }

    except Exception as e:
        logger.exception("Unexpected error while handling query.")
        return {"type": "error", "result": f"Unexpected error: {str(e)}"}

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [13]:
# Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Calculate 100 / 0",
    "Count the words in this sentence please",
    ""
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['extract', 'keywords', 'industries', 'intelligence', 'artificial']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "I don't have a specific tool for this, but here's a direct response to: 'What is machine learning?'"}
--------------------------------------------------
Query: Calculate 100 / 0
Response: {'type': 'error', 'result': 'Error in calculation'}
--------------------------------------------------
Query: Count the words in this sentence please
Response: {'type': 'count', 'result': {'word_count': 7, 'char_count': 39}}
--------------------------------------------------
Query: 
Response: {'type': 'error', 'result': 'Query must be a non-empty string.'}
-------------

In [14]:
# Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

Enter query (type 'exit' to stop): exit
